# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, roc_auc_score, classification_report, 
                             accuracy_score, ConfusionMatrixDisplay, RocCurveDisplay)
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
from catboost import CatBoostClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import warnings

warnings.filterwarnings('ignore')

# Prep

In [ ]:
#---------------------------------------------------------
# Load and Restrict to Specified Columns (without Pathology)
#---------------------------------------------------------
use_cols = ['Age','Gender','Thyroid Function','Physical Examination','Response','Response','Recurred']
df = pd.read_csv('../data/thyroid_diff.csv', usecols=use_cols)

# Label encode features with only 2 values
le = LabelEncoder()

# TODO: Input column names that only have 2 values
columns_to_encode = ['Gender']

for column in columns_to_encode:
  df[column] = le.fit_transform(df[column])

# Label encoding yes or no values
# Define mapping
mapping = {'No': 0, 'Yes': 1}

boolean_columns = ['Recurred']
df[boolean_columns] = df[boolean_columns].map(mapping.get)

# 'Physical Examination'
mapping = {'Normal': 0, 'Diffuse goiter':1, 'Single nodular goiter-left':2, 'Single nodular goiter-right':2,'Multinodular goiter':3}

df['Physical Examination'] = df['Physical Examination'].map(mapping)

if np.isnan(df['Physical Examination'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# 'Response'
mapping = {'Excellent':0, 'Indeterminate':1, 'Biochemical Incomplete':2, 'Structural Incomplete':3}

df['Response'] = df['Response'].map(mapping)

if np.isnan(df['Response'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# Initialize a OneHotEncoder object
ohe = OneHotEncoder()

# Fit the OneHotEncoder to the 'Thyroid Function' column of the dataframe 'df'
# and transform it into a one-hot encoded array. The 'toarray()' method
# converts the sparse matrix output into a dense numpy array.
ohe.fit_transform(df[['Thyroid Function']]).toarray()

# Fit the OneHotEncoder to the 'Thyroid Function' column of the dataframe 'df'
# and transform it into a one-hot encoded array. The 'toarray()' method
# converts the sparse matrix output into a dense numpy array.
feature_array = ohe.fit_transform(df[['Thyroid Function']]).toarray()

# Get the category labels from the fitted OneHotEncoder
feature_labels = ohe.categories_

# Convert the list of arrays (from ohe.categories_) to a flattened numpy array
feature_labels = np.array(feature_labels).ravel()

# Create a DataFrame from the one-hot encoded array with column names set to the flattened feature labels
pd.DataFrame(feature_array, columns=feature_labels)

# Create a DataFrame from the one-hot encoded array (feature_array)
# using the flattened feature labels (feature_labels) as column names
features = pd.DataFrame(feature_array, columns=feature_labels)

# Concatenate the original dataframe (df) and the new one-hot encoded features (features)
# along the columns (axis=1). This results in a combined DataFrame with both the original
# and one-hot encoded features.
pd.concat([df, features], axis=1)

# Concatenate the original dataframe (df) with the new one-hot encoded features (features)
# along the columns (axis=1). This adds the one-hot encoded columns to the original dataframe.
df = pd.concat([df, features], axis=1)

# Drop the original 'Thyroid Function' column from the dataframe
# as it has been replaced by the one-hot encoded columns.
df = df.drop(['Thyroid Function'], axis='columns')

#apply Box-Cox transformation to 'Age' column
age_boxcox, _ = stats.boxcox(df['Age'])

#apply Min-Max scaling to the transformed 'Age' column
scaler = MinMaxScaler()
age_scaled = scaler.fit_transform(age_boxcox.reshape(-1, 1)).flatten()

#replace 'Age' column with the scaled values
df['Age'] = age_scaled

# Drop rows with missing values
df = df.dropna()

# Identify target column
target_col = 'Recurred'
y = df[target_col]
X = df.drop(columns=[target_col])

# # Encode categorical target if needed
# if y.dtype == 'object':
#     le_target = LabelEncoder()
#     y = le_target.fit_transform(y)

# Separate numeric and categorical features
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

df.head()

# # Encode categorical features
# for cat in categorical_cols:
#     X[cat] = LabelEncoder().fit_transform(X[cat].astype(str))

# Train

In [ ]:
#---------------------------------------------------------
# Ratio-based Feature Engineering
# Numeric columns: Age, Thyroid Function, Physical Examination
#---------------------------------------------------------
def safe_div(x, y):
    return x / (y + 1e-9)

if 'Age' in numeric_cols and 'Thyroid Function' in numeric_cols:
    X['Age_ThyFun_ratio'] = safe_div(X['Age'], X['Thyroid Function'])
    X['ThyFun_Age_ratio'] = safe_div(X['Thyroid Function'], X['Age'])

if 'Physical Examination' in numeric_cols and 'Thyroid Function' in numeric_cols:
    X['PhysExam_ThyFun_ratio'] = safe_div(X['Physical Examination'], X['Thyroid Function'])
    X['ThyFun_PhysExam_ratio'] = safe_div(X['Thyroid Function'], X['Physical Examination'])

if 'Physical Examination' in numeric_cols and 'Age' in numeric_cols:
    X['PhysExam_Age_ratio'] = safe_div(X['Physical Examination'], X['Age'])
    X['Age_PhysExam_ratio'] = safe_div(X['Age'], X['Physical Examination'])

#---------------------------------------------------------
# Final Detailed Evaluation with Ratio Features
#---------------------------------------------------------
X_final = X.copy()
scaler_final = StandardScaler()
X_final_scaled = scaler_final.fit_transform(X_final)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_final_scaled, y, 
                                                            test_size=0.2, 
                                                            stratify=y, 
                                                            random_state=42)

num_classes = len(np.unique(y))
binary_classification = (num_classes == 2)

feature_names = X_final.columns  # For plotting feature importance

#---------------------------
# Logistic Regression
#---------------------------
final_lr = LogisticRegression(max_iter=1000)
final_lr.fit(X_train_f, y_train_f)
y_pred_lr = final_lr.predict(X_test_f)
y_prob_lr = final_lr.predict_proba(X_test_f)[:,1] if binary_classification else None

# Logistic Regression Feature Importance (coefficients)
if hasattr(final_lr, 'coef_'):
    lr_importances = final_lr.coef_[0]
    lr_importance_df = pd.DataFrame({'feature': feature_names, 'importance': lr_importances})
    lr_importance_df = lr_importance_df.sort_values('importance', ascending=False)
    plt.figure(figsize=(8,6))
    sns.barplot(x='importance', y='feature', data=lr_importance_df, palette='viridis')
    plt.title("Logistic Regression Feature Importance (Coefficients)")
    plt.tight_layout()
    plt.show()

#---------------------------
# XGBoost
#---------------------------
final_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
final_xgb.fit(X_train_f, y_train_f)
y_pred_xgb = final_xgb.predict(X_test_f)
y_prob_xgb = final_xgb.predict_proba(X_test_f)[:,1] if binary_classification else None

# XGBoost Feature Importance
xgb_importances = final_xgb.feature_importances_
xgb_importance_df = pd.DataFrame({'feature': feature_names, 'importance': xgb_importances})
xgb_importance_df = xgb_importance_df.sort_values('importance', ascending=False)
plt.figure(figsize=(8,6))
sns.barplot(x='importance', y='feature', data=xgb_importance_df, palette='magma')
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

#---------------------------
# CatBoost
#---------------------------
final_cat = CatBoostClassifier(verbose=False, random_state=42)
final_cat.fit(X_train_f, y_train_f)
y_pred_cat = final_cat.predict(X_test_f)
y_prob_cat = final_cat.predict_proba(X_test_f)[:,1] if binary_classification else None

# CatBoost Feature Importance
cat_importances = final_cat.get_feature_importance()
cat_importance_df = pd.DataFrame({'feature': feature_names, 'importance': cat_importances})
cat_importance_df = cat_importance_df.sort_values('importance', ascending=False)
plt.figure(figsize=(8,6))
sns.barplot(x='importance', y='feature', data=cat_importance_df, palette='cividis')
plt.title("CatBoost Feature Importance")
plt.tight_layout()
plt.show()

#---------------------------
# ANN
#---------------------------
if binary_classification:
    y_train_f_cat = y_train_f
    y_test_f_cat = y_test_f
    loss_fn = 'binary_crossentropy'
    out_dim = 1
    out_act = 'sigmoid'
else:
    y_train_f_cat = to_categorical(y_train_f, num_classes)
    y_test_f_cat = to_categorical(y_test_f, num_classes)
    loss_fn = 'categorical_crossentropy'
    out_dim = num_classes
    out_act = 'softmax'

final_ann = Sequential()
final_ann.add(Dense(32, activation='relu', input_shape=(X_train_f.shape[1],)))
final_ann.add(Dropout(0.2))
final_ann.add(Dense(16, activation='relu'))
final_ann.add(Dropout(0.2))
final_ann.add(Dense(out_dim, activation=out_act))
final_ann.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
final_ann.fit(X_train_f, y_train_f_cat, epochs=20, batch_size=16, validation_split=0.2, verbose=0)

y_pred_ann_class = final_ann.predict(X_test_f)
if binary_classification:
    y_pred_ann = (y_pred_ann_class > 0.5).astype(int).ravel()
    y_prob_ann = y_pred_ann_class.ravel()
else:
    y_pred_ann = np.argmax(y_pred_ann_class, axis=1)
    y_prob_ann = None  # AUC not straightforward for multi-class

# Note: ANN feature importance is not directly computed here.

#---------------------------------------------------------
# Print metrics and plot confusion matrix & ROC for each model
#---------------------------------------------------------
models = {
    'Logistic Regression': (y_test_f, y_pred_lr, y_prob_lr),
    'XGBoost': (y_test_f, y_pred_xgb, y_prob_xgb),
    'CatBoost': (y_test_f, y_pred_cat, y_prob_cat),
    'ANN': (y_test_f, y_pred_ann, y_prob_ann)
}

for model_name, (yt, yp, yp_prob) in models.items():
    print(f"\n=== {model_name} ===")
    print("Classification Report:")
    print(classification_report(yt, yp))
    
    acc = accuracy_score(yt, yp)
    print(f"Accuracy: {acc:.4f}")
    
    # ROC Curve / AUC (if binary)
    if binary_classification and yp_prob is not None:
        auc = roc_auc_score(yt, yp_prob)
        print(f"AUC: {auc:.4f}")
        RocCurveDisplay.from_predictions(yt, yp_prob)
        plt.title(f"{model_name} - ROC Curve (AUC = {auc:.4f})")
        plt.show()

    # Confusion Matrix
    cm = confusion_matrix(yt, yp)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"{model_name} - Confusion Matrix")
    plt.show()

# For ANN, we did not plot feature importance due to complexity.
print("\nNote: ANN feature importance not readily available without additional methods (e.g., SHAP).")